In [40]:
library(FNN)

In [41]:
source("~/bin/lit_utils.R")
lib_text()
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")

In [42]:
fread_c("../output/peaks_20250618/df1.txt") -> df1
fread_c("../output/peaks_20250618/df2.txt") -> df2

In [43]:
head(df1)
head(df2)

,mz,rt
,<dbl>,<dbl>
1,534.3142,1435.230
2,803.4847,3398.316
3,743.3755,1743.372
4,865.7914,3340.626
5,564.3281,759.228
6,668.3491,1261.596


,mz,rt
,<dbl>,<dbl>
1,452.5599,366.4339
2,430.8875,369.5029
3,670.6377,370.0573
4,389.5331,373.6752
5,389.5322,374.8447
6,603.5915,374.8447


In [44]:
nrow(df1)
nrow(df2)

[1] 85181

[1] 55973

In [84]:
ε <- 5   # 阈值
# 构造矩阵
mat1 <- as.matrix(df1[, c("mz","rt")])
mat2 <- as.matrix(df2[, c("mz","rt")])

# 查最近邻
nn <- get.knnx(data = mat1, query = mat2, k = 1)

# 拼回结果
res <- cbind(
  df2,
 index=nn$nn.index,
  df1[nn$nn.index, ],
  dist = nn$nn.dist
)
colnames(res) <- c("mz2","rt2","index_2","mz1","rt1","dist")
res %>% mutate(index_1 = row_number()) -> res

# 应用阈值
# res <- subset(res, dist <= ε)
# print(res)

In [85]:
colnames(res)

[1] "mz2"     "rt2"     "index_2" "mz1"     "rt1"     "dist"    "index_1"

In [86]:
head(res)

,mz2,rt2,index_2,mz1,rt1,dist,index_1
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<int>
588,452.5599,366.4339,588,452.5570,368.184,1.750102386,1
10907,430.8875,369.5029,10907,430.8858,369.504,0.001999725,2
45666,670.6377,370.0573,45666,670.6356,370.056,0.002452835,3
2878,389.5331,373.6752,2878,389.5321,376.188,2.512800183,4
2878.1,389.5322,374.8447,2878,389.5321,376.188,1.343300001,5
50747,603.5915,374.8447,50747,603.5897,374.844,0.001912694,6


In [87]:
nrow(res)

[1] 55973

In [70]:
fwrite_c(res,"../output/peaks_20250618/res.txt")

In [60]:
res %>% count(index) %>% filter(n>1) -> tmp

In [61]:
nrow(tmp)
head(tmp)

[1] 12570

,index,n
,<int>,<int>
1,2,2
2,9,2
3,12,2
4,18,2
5,20,3
6,24,2


In [62]:
filter(res,index %in% tmp$index) %>% head(20)

,mz,rt,index,mz2,rt2,dist
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
2878,389.5331,373.6752,2878,389.5321,376.188,2.512800183
2878.1,389.5322,374.8447,2878,389.5321,376.188,1.343300001
33350,451.2543,377.1855,33350,449.7315,378.354,1.919448263
33350.1,451.2543,378.3563,33350,449.7315,378.354,1.522791737
42557,445.7404,378.3563,42557,445.7412,379.530,1.173700301
42557.1,445.7422,379.5272,42557,445.7412,379.530,0.002960000
22858,483.2879,381.8662,22858,483.9488,381.984,0.671365580
22858.1,483.9482,381.8662,22858,483.9488,381.984,0.117801793
37689,365.8607,383.0377,37689,365.8601,383.880,0.842300207


In [63]:
max(res$dist)
min(res$dist)

[1] 4.999918

[1] 3e-05

In [64]:
summary(res)

       mz               rt             index            mz2        
 Min.   : 303.8   Min.   : 366.4   Min.   :    2   Min.   : 303.8  
 1st Qu.: 591.8   1st Qu.:1290.0   1st Qu.:18613   1st Qu.: 591.8  
 Median : 742.9   Median :2111.6   Median :38770   Median : 742.7  
 Mean   : 766.3   Mean   :2105.8   Mean   :39725   Mean   : 766.2  
 3rd Qu.: 921.1   3rd Qu.:2917.3   3rd Qu.:59842   3rd Qu.: 921.0  
 Max.   :1579.1   Max.   :3899.4   Max.   :85100   Max.   :1579.1  
      rt2              dist        
 Min.   : 368.2   Min.   :0.00003  
 1st Qu.:1290.0   1st Qu.:0.90850  
 Median :2112.3   Median :1.79580  
 Mean   :2106.3   Mean   :1.95026  
 3rd Qu.:2917.7   3rd Qu.:2.86888  
 Max.   :3899.4   Max.   :4.99992  

In [65]:
nrow(res)

[1] 52375